Cell 1 — setup

In [39]:
from pathlib import Path
import pandas as pd

PROC = Path(r"C:\Users\sclab\Desktop\A hackaton\data\processed")

master_train = pd.read_csv(PROC / "master_train.csv")
master_test  = pd.read_csv(PROC / "master_test.csv")
dictionary   = pd.read_csv(PROC / "data_dictionary_master.csv")

print("master_train:", master_train.shape)
print("master_test :", master_test.shape)
print("dictionary  :", dictionary.shape)

master_train: (15090, 25)
master_test : (3750, 24)
dictionary  : (25, 5)


In [40]:
from pathlib import Path
import pandas as pd

# Anchor to project root regardless of where the kernel starts
ROOT = Path(r"C:\Users\sclab\Desktop\A hackaton")
RAW  = ROOT / "data" / "raw"

print("RAW exists:", RAW.exists())
print("CSVs:", [f.name for f in RAW.glob("*.csv")])

RAW exists: True
CSVs: ['crop_yield_leaderboard_test.csv', 'crop_yield_train.csv', 'market_prices.csv', 'regional_weather.csv', 'submission_template.csv']


In [41]:
import os, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="colorblind")

# Find the project root by searching upward for data/raw
here = Path(os.getcwd()).resolve()
PROJECT_ROOT = None
for cand in [here, *here.parents]:
    if (cand / "data" / "raw" / "crop_yield_train.csv").exists():
        PROJECT_ROOT = cand
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Could not find project root from {here}")

raw            = PROJECT_ROOT / "data" / "raw"
processed      = PROJECT_ROOT / "data" / "processed"
reports        = PROJECT_ROOT / "reports"
figures_dir    = PROJECT_ROOT / "figures"
submission_dir = PROJECT_ROOT / "submission"
for p in [processed, reports, figures_dir, submission_dir]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("processed exists:", (processed / "master_train.csv").exists())

PROJECT_ROOT: C:\Users\sclab\Desktop\A hackaton
processed exists: True


In [42]:
from pathlib import Path
import pandas as pd

raw = Path(r"C:\Users\sclab\Desktop\A hackaton\data\raw")

prices = pd.read_csv(raw / "market_prices.csv")   # the real prices file
train  = pd.read_csv(raw / "crop_yield_train.csv")

print(prices.head())
print(train.head())

   crop_type  region  year  price_birr_per_quintal
0   sorghum   Amhara  2022                     NaN
1      wheat   SNNPR  2023                  5321.0
2      maize  Amhara  2022                  2824.0
3      Wheat  Tigray  2022                  4686.0
4    sorghum  Somali  2023                  3642.0
       plot_id  region crop_type  survey_year planting_month   altitude_m  \
0  PLOT-000339  OROMIA    barley         2024            Aug  2697.817349   
1  PLOT-007457  Amhara     Maize         2023            Jun  1523.350141   
2  PLOT-006092  OROMIA     wheat         2024            Feb  2389.441948   
3  PLOT-004128   SNNPR    barley         2022            Aug  2612.483420   
4  PLOT-003033  Amhara   sorghum         2021            Jul  1210.095040   

   rainfall_mm_season  farm_size_ha  fertilizer_kg_per_ha  improved_seed_used  \
0         1157.403560      0.867604             29.393684                   0   
1          897.420122      1.099197             16.843992            

In [43]:
import os
print("CWD:", os.getcwd())
print(os.listdir())

CWD: c:\Users\sclab\Desktop\A hackaton\notebooks\notebooks
['01_cleaning_and_integration.ipynb', '02_analysis_report.ipynb', '03_visualizations.ipynb', '04_modeling_and_evaluation.ipynb']


Cell 2 — B1.1 revenue per hectare by crop

In [44]:
price_avg = prices.groupby("crop_type")["price_birr_per_quintal"].mean()
yield_crop = train.groupby("crop_type")["yield_tons_per_ha"].mean()
rev = (yield_crop * 10 * price_avg).sort_values(ascending=False)
print("Mean yield per crop:\n", yield_crop.round(3))
print("\nMean price per crop (birr/quintal):\n", price_avg.round(1))
print("\nRevenue per hectare (birr):\n", rev.round(0))
train = pd.read_csv(processed / "master_train.csv")

Mean yield per crop:
 crop_type
 barley      2.343
 maize       3.930
 sorghum     2.692
 teff        2.040
 wheat       2.904
BARLEY       2.691
Barley       2.421
MAIZE        3.859
Maize        4.014
SORGHUM      2.600
Sorghum      2.580
TEFF         1.988
Tef          2.034
Teff         2.044
WHEAT        2.952
Wheat        2.987
barley       2.460
maize        3.899
sorghum      2.608
teff         1.998
wheat        2.763
Name: yield_tons_per_ha, dtype: float64

Mean price per crop (birr/quintal):
 crop_type
 barley      4776.0
 maize       3029.5
 sorghum     3025.2
 teff        7653.0
 wheat       5372.0
Barley       4245.0
Maize        2671.0
Sorghum      1716.5
Wheat        4686.0
barley       4107.8
maize        2968.6
sorghum      3206.6
teff         7678.6
wheat        4245.1
Name: price_birr_per_quintal, dtype: float64

Revenue per hectare (birr):
 crop_type
 teff        156120.0
 wheat       155989.0
teff         153436.0
Wheat        139973.0
 maize       119052.0
wheat 

Cell 3 — B1.2 price trend 2021–2024

In [45]:
trend = prices.groupby(["crop_type","year"])["price_birr_per_quintal"].mean().unstack()
growth = ((trend[2024] - trend[2021]) / trend[2021] * 100).sort_values(ascending=False)
print(trend.round(0))
print("\n% change 2021→2024:\n", growth.round(1))

year         2021    2022    2023    2024
crop_type                                
 barley       NaN  4179.0  4943.0  5206.0
 maize     2714.0     NaN     NaN  3345.0
 sorghum   2653.0  3302.0  3493.0     NaN
 teff         NaN  7653.0     NaN     NaN
 wheat        NaN     NaN  4846.0  5898.0
Barley     3755.0     NaN  4735.0     NaN
Maize      2576.0  2718.0     NaN     NaN
Sorghum       NaN     NaN     NaN  1716.0
Wheat         NaN  4686.0     NaN     NaN
barley     2980.0  4192.0  4723.0  4884.0
maize      2574.0  2938.0  3051.0  3205.0
sorghum    2713.0  3000.0  3471.0  3554.0
teff       6885.0  7334.0  8033.0  8572.0
wheat      3434.0  3510.0  4767.0  5472.0

% change 2021→2024:
 crop_type
barley       63.9
wheat        59.4
sorghum      31.0
maize        24.5
teff         24.5
 maize       23.2
 barley       NaN
 sorghum      NaN
 teff         NaN
 wheat        NaN
Barley        NaN
Maize         NaN
Sorghum       NaN
Wheat         NaN
dtype: float64


Cell 4 — B2.1 yield by region

In [46]:
b21 = train.groupby("region")["yield_tons_per_ha"].agg(["mean","median","std","count"])
print(b21.round(3))

         mean  median    std  count
region                             
Amhara  3.129   3.021  1.231   2979
Oromia  3.073   2.893  1.292   2969
SNNPR   3.041   2.741  1.472   2983
Somali  1.464   1.294  0.771   3069
Tigray  3.131   2.932  1.309   3090



Cell 5 — B2.2 yield by crop


In [47]:
b22 = train.groupby("crop_type")["yield_tons_per_ha"].agg(["mean","median","std","count"])
print(b22.round(3))

            mean  median    std  count
crop_type                             
barley     2.467   2.376  1.158   2934
maize      3.904   3.781  1.584   3059
sorghum    2.611   2.514  0.974   3006
teff       2.003   2.001  0.988   3015
wheat      2.802   2.692  1.411   3076


Cell 6 — B2.3 region × crop pivot

In [48]:
b23 = train.pivot_table(index="region", columns="crop_type",
                        values="yield_tons_per_ha", aggfunc="mean")
b23_counts = train.pivot_table(index="region", columns="crop_type",
                               values="yield_tons_per_ha", aggfunc="count")
print(b23.round(3))
print("\nBest cell:", b23.stack().idxmax(), "=", b23.stack().max().round(3))
print("Worst cell:", b23.stack().idxmin(), "=", b23.stack().min().round(3))

crop_type  barley  maize  sorghum   teff  wheat
region                                         
Amhara      3.187  3.829    2.495  2.479  3.598
Oromia      2.675  4.450    2.955  2.288  3.120
SNNPR       2.289  4.797    3.106  2.104  2.668
Somali      1.123  2.320    1.793  0.831  1.281
Tigray      3.010  4.074    2.709  2.332  3.463

Best cell: ('SNNPR', 'maize') = 4.797
Worst cell: ('Somali', 'teff') = 0.831


Cell 7 — B2.4 improved seed by crop

In [49]:
def gap(df, flag):
    g = df.groupby([ "crop_type", flag])["yield_tons_per_ha"].mean().unstack()
    g.columns = ["without","with"]
    g["abs_gap"] = g["with"] - g["without"]
    g["pct_lift"] = g["abs_gap"] / g["without"] * 100
    return g.round(3)
print("Improved seed:\n", gap(train, "improved_seed_used"))
print("\nPest/disease:\n", gap(train, "pest_disease_flag"))

Improved seed:
            without   with  abs_gap  pct_lift
crop_type                                   
barley       2.305  2.727    0.422    18.296
maize        3.612  4.394    0.783    21.667
sorghum      2.442  2.889    0.447    18.309
teff         1.844  2.263    0.419    22.746
wheat        2.596  3.133    0.537    20.665

Pest/disease:
            without   with  abs_gap  pct_lift
crop_type                                   
barley       2.624  1.917   -0.706   -26.922
maize        4.187  2.972   -1.215   -29.029
sorghum      2.771  2.026   -0.745   -26.896
teff         2.130  1.568   -0.562   -26.366
wheat        2.980  2.191   -0.788   -26.456


Cell 8 — B3.1 fertilizer response

In [50]:
train["fert_band"] = pd.cut(train["fertilizer_kg_per_ha"],
                            bins=[-0.01, 20, 40, 60, 10000],
                            labels=["0–20","20–40","40–60","60+"])
print(train.groupby("fert_band")["yield_tons_per_ha"].agg(["mean","count"]).round(3))
print("\nPer crop:")
print(train.pivot_table(index="fert_band", columns="crop_type",
                        values="yield_tons_per_ha", aggfunc="mean").round(3))

            mean  count
fert_band              
0–20       2.431   3666
20–40      2.712   4877
40–60      2.929   3023
60+        3.113   2776

Per crop:
crop_type  barley  maize  sorghum   teff  wheat
fert_band                                      
0–20        2.232  3.357    2.245  1.751  2.555
20–40       2.359  3.908    2.562  2.003  2.706
40–60       2.708  4.111    2.776  2.052  2.940
60+         2.759  4.376    3.027  2.262  3.143


Cell 9 — B3.2 altitude bands

In [51]:
train["alt_band"] = pd.cut(train["altitude_m"],
                           bins=[0, 1500, 2000, 2500, 3100],
                           labels=["<1500","1500–2000","2000–2500","2500+"])
piv = train.pivot_table(index="alt_band", columns="crop_type",
                        values="yield_tons_per_ha", aggfunc="mean")
cnt = train.pivot_table(index="alt_band", columns="crop_type",
                        values="yield_tons_per_ha", aggfunc="count")
print("Mean yield:\n", piv.round(3))
print("\nCounts:\n", cnt)
print("\nCells with <30 plots:\n", cnt < 30)

Mean yield:
 crop_type  barley  maize  sorghum   teff  wheat
alt_band                                       
<1500       2.634  4.020    2.696  0.953  1.872
1500–2000   0.848  3.923    2.482  2.047  0.944
2000–2500   2.100  2.122    1.364  2.285  2.734
2500+       2.657  0.478    0.298  1.300  2.978

Counts:
 crop_type  barley  maize  sorghum  teff  wheat
alt_band                                      
<1500          11   1919     2208   171     19
1500–2000      62   1009      724  1057    127
2000–2500     799    127       72  1409   1082
2500+        2062      4        2   378   1848

Cells with <30 plots:
 crop_type  barley  maize  sorghum   teff  wheat
alt_band                                       
<1500        True  False    False  False   True
1500–2000   False  False    False  False  False
2000–2500   False  False    False  False  False
2500+       False   True     True  False  False


Cell 10 — B3.3 planting month

In [52]:
order = ["Feb","Mar","Jun","Jul","Aug"]
overall = train.groupby("planting_month")["yield_tons_per_ha"].mean().reindex(order)
per_crop = train.pivot_table(index="planting_month", columns="crop_type",
                             values="yield_tons_per_ha", aggfunc="mean").reindex(order)
print("Overall:\n", overall.round(3))
print("\nPer crop:\n", per_crop.round(3))

Overall:
 planting_month
Feb    2.695
Mar    2.728
Jun    2.814
Jul    2.789
Aug    2.788
Name: yield_tons_per_ha, dtype: float64

Per crop:
 crop_type       barley  maize  sorghum   teff  wheat
planting_month                                      
Feb              2.241  4.118    2.729  1.852  2.495
Mar              2.313  3.956    2.701  1.960  2.687
Jun              2.657  3.725    2.475  2.071  3.078
Jul              2.573  3.846    2.526  2.107  2.932
Aug              2.576  3.875    2.618  2.020  2.805


Cell 11 — B3.4 distance to market

In [53]:
corr = train[["distance_to_market_km","yield_tons_per_ha"]].corr().iloc[0,1]
train["dist_band"] = pd.cut(train["distance_to_market_km"],
                            bins=[0, 5, 10, 20, 100],
                            labels=["0–5","5–10","10–20","20+"])
print("Correlation:", round(corr, 3))
print(train.groupby("dist_band")["yield_tons_per_ha"].agg(["mean","count"]).round(3))

Correlation: 0.006
            mean  count
dist_band              
0–5        2.763   5183
5–10       2.722   3397
10–20      2.788   3710
20+        2.777   2800


Cell 12 — B4.1 yield trend

In [54]:
yr = train.groupby("survey_year")["yield_tons_per_ha"].agg(["mean","std","count"])
print(yr.round(3))
print("\nBy region:")
print(train.pivot_table(index="survey_year", columns="region",
                        values="yield_tons_per_ha", aggfunc="mean").round(3))

              mean    std  count
survey_year                     
2021         2.700  1.400   3738
2022         2.778  1.377   3780
2023         2.793  1.386   3759
2024         2.779  1.434   3813

By region:
region       Amhara  Oromia  SNNPR  Somali  Tigray
survey_year                                       
2021          3.146   3.041  2.864   1.283   3.159
2022          3.080   3.031  3.024   1.671   3.139
2023          3.105   3.132  3.161   1.415   3.155
2024          3.181   3.089  3.105   1.478   3.068


In [55]:
print("train cols:", list(train.columns))
print("has season_mean_temp:", "season_mean_temp" in train.columns)
print("has season_total_rain:", "season_total_rain" in train.columns)

train cols: ['plot_id', 'region', 'crop_type', 'survey_year', 'planting_month', 'altitude_m', 'rainfall_mm_season', 'farm_size_ha', 'fertilizer_kg_per_ha', 'improved_seed_used', 'pest_disease_flag', 'soil_quality_index', 'labor_days_per_ha', 'distance_to_market_km', 'yield_tons_per_ha', 'planting_month_num', 'season_mean_temp', 'season_total_rain', 'season_extreme_heat', 'season_months_count', 'region_year_temp', 'region_year_rain', 'temp_dev', 'rain_dev', 'fert_x_seed', 'fert_band', 'alt_band', 'dist_band']
has season_mean_temp: True
has season_total_rain: True


Cell 13 — B4.2 weather anomalies

In [56]:
reg_year = train.groupby(["region","survey_year"]).agg(
    temp=("season_mean_temp","mean"),
    rain=("season_total_rain","mean"),
    y=("yield_tons_per_ha","mean")).reset_index()
reg_year["temp_dev"] = reg_year["temp"] - reg_year.groupby("region")["temp"].transform("mean")
print("Most unusual region-years (temp):")
print(reg_year.reindex(reg_year["temp_dev"].abs().sort_values(ascending=False).index).head(5).round(3))

print("\nCrop-level check for that region-year:")
worst = reg_year.loc[reg_year["temp_dev"].abs().idxmax()]
sub = train[(train["region"]==worst["region"]) & (train["survey_year"]==worst["survey_year"])]
print(sub.groupby("crop_type")["yield_tons_per_ha"].mean().round(3))

Most unusual region-years (temp):
    region  survey_year    temp     rain      y  temp_dev
19  Tigray         2024  20.303  220.193  3.068     2.569
13  Somali         2022  25.925  179.478  1.671    -2.475
12  Somali         2021  30.145  141.875  1.283     1.745
1   Amhara         2022  18.406  250.887  3.080     1.598
17  Tigray         2022  16.344  424.567  3.139    -1.390

Crop-level check for that region-year:
crop_type
barley     2.326
maize      4.949
sorghum    3.301
teff       2.061
wheat      2.608
Name: yield_tons_per_ha, dtype: float64


Cell 14 — B4.3 plot vs station rainfall

In [57]:
comp = train.dropna(subset=["rainfall_mm_season","season_total_rain"])
corr = comp[["rainfall_mm_season","season_total_rain"]].corr().iloc[0,1]
mad  = (comp["rainfall_mm_season"] - comp["season_total_rain"]).abs().mean()
print(f"Correlation: {corr:.3f}")
print(f"Mean absolute difference: {mad:.1f} mm")
print(f"Plot mean: {comp['rainfall_mm_season'].mean():.1f} mm")
print(f"Station mean: {comp['season_total_rain'].mean():.1f} mm")

Correlation: 0.005
Mean absolute difference: 499.8 mm
Plot mean: 851.3 mm
Station mean: 367.3 mm
